# BWI Instagram Content Analytics (*Jan 2025 - Sep 2026*)

*@ Wiraatmaja*

This is the follow-up to my earlier *BWI Instagram & Meta Ads Analysis (2022-Feb 2025)*. That notebook ended in February 2025; In this project, I helped Business Wisdom Institute (BWI) log every Instagram post in a new Google Sheet, so this notebook picks up from January 2025 and runs through September 2026.

This time I did the cleaning and analysis in **SQL (DuckDB)** inside Jupyter, using Python only to read the file and reshape the sheet into rows. The questions:

1. *Which formats, content types and posting days earn the most engagement?*
2. *How has engagement moved month to month?*
3. *Which collaborators (speakers, trainers, guests) bring the most engagement when they appear in a post?*

**Tools:** Python (*pandas, re*), DuckDB SQL (*CTEs, window functions, regex, UNNEST*)

## 1. Import & Data Loading

The sheet is exported as a CSV with no usable header row, so I load it with `header=None` and look at the first rows before deciding how to parse it.

In [16]:
import pandas as pd
import re

raw = pd.read_csv("Content Analytics Updated - Analytics.csv", header=None)
print(raw.shape)
raw.head(10)

(498, 21)


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,September 2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Day,Date,Time,Content,Type,Slides,Facebook,NaN,NaN,NaN,...,NaN,Instagram,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,Reach,Engagement,NaN,NaN,...,Engagement in total,Total Views,Account Reach,Engagement,NaN,NaN,NaN,Engagement in total,Average watch time (sec),Duration
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reactions,Comments,Shares,...,NaN,NaN,NaN,Likes,Comments,Shares,Saves,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Tuesday,01/09/2026,NaN,24/9/26 Personal Branding Promo,Promotionals,Video,NaN,NaN,NaN,NaN,...,NaN,415,293,14,0,2,0,16,10,00:34
6,Thursday,03/09/2026,NaN,Podcast - Leverage Talk Collab w/ Edward Santoso,"Entertainment, Educational",Video,NaN,NaN,NaN,NaN,...,NaN,6727,4715,161,10,52,89,312,15,00:51
7,Thursday,03/09/2026,NaN,IFG x BWI Recap post,Documentation,10,NaN,NaN,NaN,NaN,...,NaN,376,154,17,0,0,0,17,Not-Applicable,-
8,Saturday,05/09/2026,NaN,Podcast - External Factor Talk Collab w/ Edwar...,Documentation,Video,NaN,NaN,NaN,NaN,...,NaN,1837,1105,46,10,8,7,71,10,01:31
9,Saturday,05/09/2026,NaN,TS Storytelling - You are your own choices,Educational,Video,NaN,NaN,NaN,NaN,...,NaN,5723,4447,183,2,138,27,350,01:34,02:55


The raw file is not one table. It is **21 monthly blocks stacked on top of each other**: each block starts with a month label (`September 2026`), then a 4-row merged header (Day / Date / Content … then Facebook and Instagram metric groups), then that month's posts. 498 raw rows, but many are labels, headers or blank spacers.

So instead of a fixed `skiprows`, I walk through the rows one by one:
- a row whose first cell matches `<Month> <Year>` and whose second cell is empty starts a new block;
- the `Day` header line right after it is skipped;
- fully blank rows (no Day and no Date) are skipped;
- everything else is a real post and is kept.

In [17]:
import re

month_pat = re.compile(r'^(January|February|March|April|May|June|July|August|September|October|November|December) \d{4}$')

def is_month_marker(row):
    v0, v1 = row[0], row[1]
    return isinstance(v0, str) and bool(month_pat.match(v0.strip())) and pd.isna(v1)

rows = raw.values.tolist()
records = []
i = 0
n = len(rows)

while i < n:
    row = rows[i]
    if is_month_marker(row):
        i += 1
        if i < n and rows[i][0] == 'Day':
            i += 1
        while i < n:
            r = rows[i]
            if is_month_marker(r):
                break
            if pd.isna(r[0]) and pd.isna(r[1]):
                i += 1
                continue
            records.append(r[:21])
            i += 1
        continue
    i += 1

print("total rows parsed:", len(records))

total rows parsed: 390


**390 real posts** out of 498 raw rows. Now I give those rows proper column names. The 21 columns follow the sheet's layout: post info, 6 Facebook metrics, then 9 Instagram metrics. I also add a `raw_row_id` so every post has a stable key I can join on later.

In [18]:
import pandas as pd

posts_raw = pd.DataFrame(records, columns=[
    "day", "date", "post_time", "content", "type_raw", "slides_raw",
    "fb_reach", "fb_reactions", "fb_comments", "fb_shares", "fb_saves", "fb_engagement_total",
    "ig_total_views", "ig_account_reach", "ig_likes", "ig_comments", "ig_shares", "ig_saves",
    "ig_engagement_total", "ig_avg_watch_time_raw", "ig_duration_raw",
])
posts_raw.insert(0, "raw_row_id", range(1, len(posts_raw) + 1))

print(posts_raw.shape)
posts_raw.head(5)

(390, 22)


,raw_row_id,day,date,post_time,content,type_raw,slides_raw,fb_reach,fb_reactions,fb_comments,...,fb_engagement_total,ig_total_views,ig_account_reach,ig_likes,ig_comments,ig_shares,ig_saves,ig_engagement_total,ig_avg_watch_time_raw,ig_duration_raw
0,1,Tuesday,01/09/2026,NaN,24/9/26 Personal Branding Promo,Promotionals,Video,NaN,NaN,NaN,...,NaN,415,293,14,0,2,0,16,10,00:34
1,2,Thursday,03/09/2026,NaN,Podcast - Leverage Talk Collab w/ Edward Santoso,"Entertainment, Educational",Video,NaN,NaN,NaN,...,NaN,6727,4715,161,10,52,89,312,15,00:51
2,3,Thursday,03/09/2026,NaN,IFG x BWI Recap post,Documentation,10,NaN,NaN,NaN,...,NaN,376,154,17,0,0,0,17,Not-Applicable,-
3,4,Saturday,05/09/2026,NaN,Podcast - External Factor Talk Collab w/ Edwar...,Documentation,Video,NaN,NaN,NaN,...,NaN,1837,1105,46,10,8,7,71,10,01:31
4,5,Saturday,05/09/2026,NaN,TS Storytelling - You are your own choices,Educational,Video,NaN,NaN,NaN,...,NaN,5723,4447,183,2,138,27,350,01:34,02:55


## 2. Moving into SQL

From here on, everything runs in DuckDB. I register the pandas DataFrame as a SQL table called `posts_raw` and run a first sanity check: every row should have content.

In [19]:
import duckdb

con = duckdb.connect(":memory:")
con.register("posts_raw", posts_raw)

con.sql("SELECT COUNT(*) AS n_rows, COUNT(content) AS n_with_content FROM posts_raw").show()

┌────────┬────────────────┐
│ n_rows │ n_with_content │
│ int64  │     int64      │
├────────┼────────────────┤
│    390 │            390 │
└────────┴────────────────┘



### 2.1 Fixing types

Every column came in as text. Dates are written day-first (`01/09/2026` = 1 September), so I test `STRPTIME` with the `%d/%m/%Y` pattern on a few rows before applying it to the whole table.

In [20]:
con.sql("""
    SELECT 
        date AS date_text,
        STRPTIME(date, '%d/%m/%Y') AS date_parsed
    FROM posts_raw
    LIMIT 5
""").show()

┌────────────┬─────────────────────┐
│ date_text  │     date_parsed     │
│  varchar   │      timestamp      │
├────────────┼─────────────────────┤
│ 01/09/2026 │ 2026-09-01 00:00:00 │
│ 03/09/2026 │ 2026-09-03 00:00:00 │
│ 03/09/2026 │ 2026-09-03 00:00:00 │
│ 05/09/2026 │ 2026-09-05 00:00:00 │
│ 05/09/2026 │ 2026-09-05 00:00:00 │
└────────────┴─────────────────────┘



The date pattern works. Now I build the first cleaned stage:
- `date` becomes a real `DATE`;
- every metric goes through `TRY_CAST(... AS INTEGER)`, which returns `NULL` instead of crashing when a cell holds text such as `-`;
- `post_time` is dropped because it was never filled in;
- the messy text columns (`type_raw`, `slides_raw`, watch time, duration) are kept as text for now, since they need their own cleanup.

I build the cleaning in stages (`stage1` -> `stage2` -> `stage3` -> `posts`) so each step can be checked on its own.

In [21]:
con.sql("""
    CREATE TABLE posts_stage1 AS
    SELECT
        raw_row_id,
        day,
        STRPTIME(date, '%d/%m/%Y')::DATE AS date,
        content,
        type_raw,
        slides_raw,
        TRY_CAST(fb_reach AS INTEGER) AS fb_reach,
        TRY_CAST(fb_reactions AS INTEGER) AS fb_reactions,
        TRY_CAST(fb_comments AS INTEGER) AS fb_comments,
        TRY_CAST(fb_shares AS INTEGER) AS fb_shares,
        TRY_CAST(fb_saves AS INTEGER) AS fb_saves,
        TRY_CAST(fb_engagement_total AS INTEGER) AS fb_engagement_total,
        TRY_CAST(ig_total_views AS INTEGER) AS ig_total_views,
        TRY_CAST(ig_account_reach AS INTEGER) AS ig_account_reach,
        TRY_CAST(ig_likes AS INTEGER) AS ig_likes,
        TRY_CAST(ig_comments AS INTEGER) AS ig_comments,
        TRY_CAST(ig_shares AS INTEGER) AS ig_shares,
        TRY_CAST(ig_saves AS INTEGER) AS ig_saves,
        TRY_CAST(ig_engagement_total AS INTEGER) AS ig_engagement_total,
        ig_avg_watch_time_raw,
        ig_duration_raw
    FROM posts_raw
""")

con.sql("SELECT COUNT(*) FROM posts_stage1").show()
con.sql("SELECT * FROM posts_stage1 LIMIT 5").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          390 │
└──────────────┘

┌────────────┬──────────┬────────────┬──────────────────────────────────────────────────────────┬────────────────────────────┬────────────┬──────────┬──────────────┬─────────────┬───────────┬──────────┬─────────────────────┬────────────────┬──────────────────┬──────────┬─────────────┬───────────┬──────────┬─────────────────────┬───────────────────────┬─────────────────┐
│ raw_row_id │   day    │    date    │                         content                          │          type_raw          │ slides_raw │ fb_reach │ fb_reactions │ fb_comments │ fb_shares │ fb_saves │ fb_engagement_total │ ig_total_views │ ig_account_reach │ ig_likes │ ig_comments │ ig_shares │ ig_saves │ ig_engagement_total │ ig_avg_watch_time_raw │ ig_duration_raw │
│   int64    │ varchar  │    date    │                         varchar                          │          varchar           │  varchar   │  int32   │ 

Same result viewed as a DataFrame. Note that every Facebook column is `NULL`: BWI never logged Facebook numbers, so the rest of the analysis is Instagram only.

In [23]:
con.sql("SELECT * FROM posts_stage1 LIMIT 5").df()

,raw_row_id,day,date,content,type_raw,slides_raw,fb_reach,fb_reactions,fb_comments,fb_shares,...,fb_engagement_total,ig_total_views,ig_account_reach,ig_likes,ig_comments,ig_shares,ig_saves,ig_engagement_total,ig_avg_watch_time_raw,ig_duration_raw
0,1,Tuesday,2026-09-01,24/9/26 Personal Branding Promo,Promotionals,Video,<NA>,<NA>,<NA>,<NA>,...,<NA>,415,293,14,0,2,0,16,10,00:34
1,2,Thursday,2026-09-03,Podcast - Leverage Talk Collab w/ Edward Santoso,"Entertainment, Educational",Video,<NA>,<NA>,<NA>,<NA>,...,<NA>,6727,4715,161,10,52,89,312,15,00:51
2,3,Thursday,2026-09-03,IFG x BWI Recap post,Documentation,10,<NA>,<NA>,<NA>,<NA>,...,<NA>,376,154,17,0,0,0,17,Not-Applicable,-
3,4,Saturday,2026-09-05,Podcast - External Factor Talk Collab w/ Edwar...,Documentation,Video,<NA>,<NA>,<NA>,<NA>,...,<NA>,1837,1105,46,10,8,7,71,10,01:31
4,5,Saturday,2026-09-05,TS Storytelling - You are your own choices,Educational,Video,<NA>,<NA>,<NA>,<NA>,...,<NA>,5723,4447,183,2,138,27,350,01:34,02:55


### 2.2 Profiling the messy columns

Before writing any `CASE` logic I list the distinct values in each text column, so the cleanup is based on what is actually there rather than guesses.

Slides: a number means a carousel with that many slides, the word *Video* means a Reel, and `NULL` means a single image. *Video* is spelled three ways (`Video`, `video`, `VIdeo`).

In [25]:
con.sql("SELECT DISTINCT slides_raw FROM posts_stage1 ORDER BY slides_raw").show()

┌────────────┐
│ slides_raw │
│  varchar   │
├────────────┤
│ 1          │
│ 10         │
│ 11         │
│ 12         │
│ 13         │
│ 14         │
│ 15         │
│ 16         │
│ 17         │
│ 18         │
│ ·          │
│ ·          │
│ ·          │
│ 4          │
│ 5          │
│ 6          │
│ 7          │
│ 8          │
│ 9          │
│ VIdeo      │
│ Video      │
│ video      │
│ NULL       │
└────────────┘
   26 rows  
  (20 shown) 



Type: much cleaner than the 2022-2025 sheet (no abbreviations or Indonesian labels this time), but some posts carry two tags separated by a comma, e.g. `Documentation, Educational`. I handle that in section 2.4.

In [26]:
con.sql("SELECT DISTINCT type_raw FROM posts_stage1 ORDER BY type_raw").show()

┌─────────────────────────────┐
│          type_raw           │
│           varchar           │
├─────────────────────────────┤
│ BWI                         │
│ BWI, Documentation          │
│ BWI, Greetings              │
│ Conversational              │
│ Documentation               │
│ Documentation, BWI          │
│ Documentation, Educational  │
│ Documentation, Promotionals │
│ Educational                 │
│ Educational, Entertainment  │
│ Educational, Promotionals   │
│ Entertainment               │
│ Entertainment, Educational  │
│ Greetings                   │
│ Promotionals                │
│ Promotionals, Educational   │
└─────────────────────────────┘
            16 rows          



Average watch time: mixes plain seconds (`15`), minute format (`01:34`), and several "no value" labels (`-`, `--`, `Not shown`, `Not-Applicable`, `Non-applicable`). It also contains `Boosted` (and a typo, `Boosed`): the team wrote that in place of the watch time when the post was a paid boost.

In [27]:
con.sql("SELECT DISTINCT ig_avg_watch_time_raw FROM posts_stage1 ORDER BY ig_avg_watch_time_raw").show()

┌───────────────────────┐
│ ig_avg_watch_time_raw │
│        varchar        │
├───────────────────────┤
│ -                     │
│ --                    │
│ 01:34                 │
│ 01:41                 │
│ 10                    │
│ 11                    │
│ 12                    │
│ 13                    │
│ 14                    │
│ 15                    │
│ ·                     │
│ ·                     │
│ ·                     │
│ 7                     │
│ 8                     │
│ 9                     │
│ Boosed                │
│ Boosted               │
│ Non-applicable        │
│ Not Shown             │
│ Not shown             │
│ Not-Applicable        │
│ NULL                  │
└───────────────────────┘
   35 rows (20 shown)  



**Duration:** always `mm:ss` for videos, `-` for non-video posts.

In [28]:
con.sql("SELECT DISTINCT ig_duration_raw FROM posts_stage1 ORDER BY ig_duration_raw").show()

┌─────────────────┐
│ ig_duration_raw │
│     varchar     │
├─────────────────┤
│ -               │
│ 00:19           │
│ 00:20           │
│ 00:21           │
│ 00:22           │
│ 00:23           │
│ 00:24           │
│ 00:25           │
│ 00:26           │
│ 00:27           │
│   ·             │
│   ·             │
│   ·             │
│ 02:09           │
│ 02:19           │
│ 02:24           │
│ 02:37           │
│ 02:55           │
│ 02:56           │
│ 03:26           │
│ 03:31           │
│ 05:53           │
│ NULL            │
└─────────────────┘
      87 rows    
    (20 shown)    



### 2.3 Format, watch time and duration

Format: I lower-case and trim `slides_raw` so all three spellings of *Video* match, then sort every post into Video, Carousel or Single Image. The slide count is kept as its own column.

In [29]:
con.sql("""
    CREATE TABLE posts_stage2 AS
    SELECT
        *,
        CASE
            WHEN LOWER(TRIM(slides_raw)) = 'video' THEN 'Video'
            WHEN TRY_CAST(slides_raw AS INTEGER) IS NOT NULL THEN 'Carousel'
            WHEN slides_raw IS NULL THEN 'Single Image'
        END AS format,
        TRY_CAST(slides_raw AS INTEGER) AS slide_count
    FROM posts_stage1
""")

con.sql("SELECT format, COUNT(*) FROM posts_stage2 GROUP BY format").show()

┌──────────────┬──────────────┐
│    format    │ count_star() │
│   varchar    │    int64     │
├──────────────┼──────────────┤
│ Carousel     │          189 │
│ Video        │          189 │
│ Single Image │           12 │
└──────────────┴──────────────┘



Watch time and duration are converted to seconds:
- anything with a colon is split into minutes and seconds (`01:34` → 94);
- plain numbers are already seconds;
- every "no value" label becomes `NULL`.

The `Boosted` labels are not thrown away: they become an `is_boosted` flag, because a paid boost inflates reach and is worth knowing about.

In [30]:
con.sql("""
    CREATE TABLE posts_stage3 AS
    SELECT
        *,
        LOWER(TRIM(ig_avg_watch_time_raw)) IN ('boosted', 'boosed') AS is_boosted,
        CASE
            WHEN LOWER(TRIM(ig_avg_watch_time_raw)) IN ('boosted', 'boosed') THEN NULL
            WHEN LOWER(TRIM(ig_avg_watch_time_raw)) IN ('not-applicable', 'non-applicable', 'not shown', '-', '--') THEN NULL
            WHEN ig_avg_watch_time_raw LIKE '%:%' THEN
                CAST(SPLIT_PART(ig_avg_watch_time_raw, ':', 1) AS INTEGER) * 60
                + CAST(SPLIT_PART(ig_avg_watch_time_raw, ':', 2) AS INTEGER)
            ELSE TRY_CAST(ig_avg_watch_time_raw AS INTEGER)
        END AS ig_avg_watch_time_sec,
        CASE
            WHEN TRIM(ig_duration_raw) = '-' THEN NULL
            WHEN ig_duration_raw LIKE '%:%' THEN
                CAST(SPLIT_PART(ig_duration_raw, ':', 1) AS INTEGER) * 60
                + CAST(SPLIT_PART(ig_duration_raw, ':', 2) AS INTEGER)
            ELSE TRY_CAST(ig_duration_raw AS INTEGER)
        END AS ig_duration_sec
    FROM posts_stage2
""")

con.sql("""
    SELECT ig_avg_watch_time_raw, is_boosted, ig_avg_watch_time_sec, ig_duration_raw, ig_duration_sec
    FROM posts_stage3
    LIMIT 10
""").df()

,ig_avg_watch_time_raw,is_boosted,ig_avg_watch_time_sec,ig_duration_raw,ig_duration_sec
0,10,False,10,00:34,34
1,15,False,15,00:51,51
2,Not-Applicable,False,<NA>,-,<NA>
3,10,False,10,01:31,91
4,01:34,False,94,02:55,175
5,18,False,18,00:33,33
6,8,False,8,00:59,59
7,Not-Applicable,False,<NA>,-,<NA>
8,14,False,14,00:50,50
9,34,False,34,01:41,101


Checking the flag: every `Boosted`/`Boosed` row is marked `True`, with no watch time

In [31]:
con.sql("""
    SELECT ig_avg_watch_time_raw, is_boosted, ig_avg_watch_time_sec
    FROM posts_stage3
    WHERE is_boosted = TRUE
""").df()

,ig_avg_watch_time_raw,is_boosted,ig_avg_watch_time_sec
0,Boosted,True,<NA>
1,Boosted,True,<NA>
2,Boosted,True,<NA>
3,Boosted,True,<NA>
4,Boosed,True,<NA>
5,Boosted,True,<NA>
6,Boosted,True,<NA>
7,Boosted,True,<NA>
8,Boosted,True,<NA>
9,Boosted,True,<NA>


10 posts were boosted They stay in the dataset, but their reach was bought, which is worth keeping in mind when reading the reach columns below

In [32]:
con.sql("SELECT COUNT(*) AS boosted_count FROM posts_stage3 WHERE is_boosted = TRUE").show()

┌───────────────┐
│ boosted_count │
│     int64     │
├───────────────┤
│            10 │
└───────────────┘



### 2.4 Splitting multi-tagged content types

A post tagged `Documentation, Educational` belongs to both categories. In the 2022-2025 notebook I kept only the first tag; here I keep both, which is fairer to the second tag.

To do that I build a separate *bridge table*, `post_types`: `STRING_SPLIT` turns the tag text into a list and `UNNEST` gives each tag its own row. One post with two tags becomes two rows, both pointing to the same `raw_row_id`

In [33]:
con.sql("""
    CREATE TABLE post_types AS
    SELECT
        raw_row_id,
        TRIM(UNNEST(STRING_SPLIT(type_raw, ','))) AS content_type
    FROM posts_stage3
""")

con.sql("SELECT COUNT(*) AS total_tag_rows FROM post_types").show()
con.sql("SELECT content_type, COUNT(*) FROM post_types GROUP BY content_type ORDER BY COUNT(*) DESC").df()

┌────────────────┐
│ total_tag_rows │
│     int64      │
├────────────────┤
│            424 │
└────────────────┘



,content_type,count_star()
0,Promotionals,157
1,Documentation,146
2,Educational,65
3,Greetings,25
4,Entertainment,21
5,BWI,7
6,Conversational,3


390 posts produce **424 tag rows**, so 34 posts had two tags. Seven clean categories remain.

### 2.5 Final clean table

The last stage drops the raw text columns I've now replaced (`EXCLUDE` keeps everything else) and saves the result as `posts`: one row per post, all typed correctly.

In [34]:
con.sql("""
    CREATE TABLE posts AS
    SELECT * EXCLUDE (type_raw, slides_raw, ig_avg_watch_time_raw, ig_duration_raw)
    FROM posts_stage3
""")

con.sql("SELECT COUNT(*) FROM posts").show()
con.sql("DESCRIBE posts").df()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          390 │
└──────────────┘



,column_name,column_type,null,key,default,extra
0,raw_row_id,BIGINT,YES,None,None,None
1,day,VARCHAR,YES,None,None,None
2,date,DATE,YES,None,None,None
3,content,VARCHAR,YES,None,None,None
4,fb_reach,INTEGER,YES,None,None,None
5,fb_reactions,INTEGER,YES,None,None,None
6,fb_comments,INTEGER,YES,None,None,None
7,fb_shares,INTEGER,YES,None,None,None
8,fb_saves,INTEGER,YES,None,None,None
9,fb_engagement_total,INTEGER,YES,None,None,None


## 3. Analysis

With one clean `posts` table and the `post_types` bridge table ready, every question below is answered with a single SQL query.

### 3.1 Monthly overview

Post count, average reach and average engagement (*likes + comments + shares + saves*) for each month. `DATE_TRUNC('month', date)` groups every post by the first day of its month.

In [35]:
con.sql("""
    SELECT
        DATE_TRUNC('month', date) AS month,
        COUNT(*) AS num_posts,
        ROUND(AVG(ig_account_reach), 0) AS avg_reach,
        ROUND(AVG(ig_engagement_total), 0) AS avg_engagement
    FROM posts
    GROUP BY DATE_TRUNC('month', date)
    ORDER BY month
""").df()

,month,num_posts,avg_reach,avg_engagement
0,2025-01-01,21,1007.0,20.0
1,2025-02-01,19,896.0,14.0
2,2025-03-01,19,1463.0,41.0
3,2025-04-01,19,554.0,20.0
4,2025-05-01,19,2597.0,133.0
5,2025-06-01,16,3887.0,66.0
6,2025-07-01,5,2376.0,66.0
7,2025-08-01,31,1166.0,32.0
8,2025-09-01,24,467.0,19.0
9,2025-10-01,33,1057.0,30.0


Two months stand out: **May 2025** (*avg. 133 engagements, reach ~2,600*) and **March 2026** (*avg 89*). September 2026 also jumped to 82. Engagement otherwise sits mostly between 15 and 50, while post volume swings a lot (5 posts in July 2025, 33 in October 2025).

### 3.2 Performance by content type

Here I join `posts` to the `post_types` bridge table, so a double-tagged post counts once in each of its categories.

In [36]:
con.sql("""
    SELECT
        pt.content_type,
        COUNT(*) AS num_posts,
        ROUND(AVG(p.ig_account_reach), 0) AS avg_reach,
        ROUND(AVG(p.ig_engagement_total), 0) AS avg_engagement
    FROM posts p
    JOIN post_types pt ON p.raw_row_id = pt.raw_row_id
    GROUP BY pt.content_type
    ORDER BY avg_engagement DESC
""").df()

,content_type,num_posts,avg_reach,avg_engagement
0,BWI,7,1153.0,82.0
1,Educational,65,900.0,61.0
2,Promotionals,157,1808.0,51.0
3,Entertainment,21,913.0,41.0
4,Documentation,146,493.0,27.0
5,Conversational,3,235.0,9.0
6,Greetings,25,174.0,9.0


- `Educational` content is the strongest category with a real sample size: avg 61 engagements over 65 posts.
- `Promotional` posts reach the most people (avg 1,808), but their engagement (51) does not keep pace with that reach, which matches what I found in 2022-2025.
- `Documentation` is the second most-posted category (146) but sits near the bottom (27).
- `Greetings` and `Conversational` posts earn almost nothing (9).
- `BWI` posts top the table at 82, but with only 7 posts that is not reliable on its own.

### 3.3 Performance by format

`HAVING COUNT(*) >= 10` keeps only formats with enough posts to compare.

In [37]:
con.sql("""
    SELECT
        format,
        COUNT(*) AS num_posts,
        ROUND(AVG(ig_account_reach), 0) AS avg_reach,
        ROUND(AVG(ig_engagement_total), 0) AS avg_engagement
    FROM posts
    GROUP BY format
    HAVING COUNT(*) >= 10
    ORDER BY avg_engagement DESC
""").df()

,format,num_posts,avg_reach,avg_engagement
0,Video,189,1699.0,63.0
1,Carousel,189,491.0,19.0
2,Single Image,12,414.0,18.0


This is the biggest gap in the dataset: Video averages **63 engagements and 1,699 reach**, against **19 and 491 for Carousels**, with exactly 189 posts each. Single images perform about the same as carousels.

This differs from the 2022-2025 notebook, where video and image engagement *rates* were almost identical. The likely reason: this notebook compares raw totals, and videos reach far more people, so they collect more engagement in absolute numbers. (See the note on rates in section 5.)

### 3.4 Performance by posting day

`STRFTIME(date, '%A')` turns each date into its weekday name.

In [38]:
con.sql("""
    SELECT
        STRFTIME(date, '%A') AS weekday,
        COUNT(*) AS num_posts,
        ROUND(AVG(ig_account_reach), 0) AS avg_reach,
        ROUND(AVG(ig_engagement_total), 0) AS avg_engagement
    FROM posts
    GROUP BY weekday
    ORDER BY avg_engagement DESC
""").df()

,weekday,num_posts,avg_reach,avg_engagement
0,Sunday,39,648.0,56.0
1,Wednesday,60,836.0,50.0
2,Saturday,51,1067.0,46.0
3,Monday,71,1094.0,41.0
4,Thursday,55,876.0,34.0
5,Friday,57,1425.0,32.0
6,Tuesday,57,1425.0,30.0


Two different winners depending on the measure:
- **Engagement** is highest on *Sunday* (56) and Wednesday (50).
- **Reach** is highest on *Tuesday and Friday* (1,425), yet those two days have the *lowest* engagement.

Days that reach more people are not the days people interact most.

### 3.5 Top 3 posts in each content type

A window function, `RANK() OVER (PARTITION BY content_type ORDER BY engagement DESC)`, ranks posts *within* each category; then I keep ranks 1-3.

In [39]:
con.sql("""
    WITH ranked AS (
        SELECT
            pt.content_type,
            p.content,
            p.ig_engagement_total,
            RANK() OVER (PARTITION BY pt.content_type ORDER BY p.ig_engagement_total DESC) AS rank_in_category
        FROM posts p
        JOIN post_types pt ON p.raw_row_id = pt.raw_row_id
    )
    SELECT *
    FROM ranked
    WHERE rank_in_category <= 3
    ORDER BY content_type, rank_in_category
""").df()

,content_type,content,ig_engagement_total,rank_in_category
0,BWI,TS Birthday highlight & Recap,129,1
1,BWI,Idul Fitri Interactive Post w./ BWI Internal Team,114,2
2,BWI,Honda x BWI appreciation post,113,3
3,Conversational,Podcast Highlights #6,17,1
4,Conversational,New Year Check-In,6,2
5,Conversational,Produktif atau sibuk,5,3
6,Documentation,24/4/25 Negotiation Training Recap Collab w/ Ivi,1112,1
7,Documentation,Personal branding Seminar Recap Collab w/ Ivi,114,2
8,Documentation,Honda x BWI appreciation post,113,3
9,Educational,TS Storytelling and Indonesia Sales Festival P...,458,1


Patterns in the top posts:
- **Collaborations dominate.** Almost every top post is a podcast, recap or promo featuring a named guest or trainer.
- The single biggest post is the *Negotiation Training Recap with Ivi (1,112 engagements)*, about 8× the next Documentation post. One outlier like this can pull a category's average up on its own.
- **Storytelling and podcasts** lead Educational and Entertainment.

### 3.6 Month-over-month change

`LAG()` looks at the previous row in date order, so I can show each month's change in average engagement next to the month before it.

In [40]:
con.sql("""
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', date) AS month,
            COUNT(*) AS num_posts,
            ROUND(AVG(ig_engagement_total), 0) AS avg_engagement
        FROM posts
        GROUP BY DATE_TRUNC('month', date)
    )
    SELECT
        month,
        num_posts,
        avg_engagement,
        LAG(avg_engagement) OVER (ORDER BY month) AS prev_month_engagement,
        avg_engagement - LAG(avg_engagement) OVER (ORDER BY month) AS change
    FROM monthly
    ORDER BY month
""").df()

,month,num_posts,avg_engagement,prev_month_engagement,change
0,2025-01-01,21,20.0,NaN,NaN
1,2025-02-01,19,14.0,20.0,-6.0
2,2025-03-01,19,41.0,14.0,27.0
3,2025-04-01,19,20.0,41.0,-21.0
4,2025-05-01,19,133.0,20.0,113.0
5,2025-06-01,16,66.0,133.0,-67.0
6,2025-07-01,5,66.0,66.0,0.0
7,2025-08-01,31,32.0,66.0,-34.0
8,2025-09-01,24,19.0,32.0,-13.0
9,2025-10-01,33,30.0,19.0,11.0


The swings are sharp (+113 into May 2025, −67 the month after; +76 into March 2026, −56 after). Engagement doesn't build up month after month; it **spikes around specific events and drops back**. That makes the calendar of seminars and campaigns a bigger driver than any steady trend.

### 3.7 Summary: the best of each dimension

One query that stacks the winner of each earlier breakdown with `UNION ALL`. Each part is wrapped in brackets so its own `ORDER BY … LIMIT 1` applies only to that part.

In [42]:
con.sql("""
    (SELECT 'Best Format' AS dimension, format AS top_performer, ROUND(AVG(ig_engagement_total), 0) AS avg_engagement, COUNT(*) AS n
    FROM posts
    GROUP BY format
    ORDER BY avg_engagement DESC
    LIMIT 1)

    UNION ALL

    (SELECT 'Best Content Type', pt.content_type, ROUND(AVG(p.ig_engagement_total), 0), COUNT(*)
    FROM posts p
    JOIN post_types pt ON p.raw_row_id = pt.raw_row_id
    GROUP BY pt.content_type
    ORDER BY 3 DESC
    LIMIT 1)

    UNION ALL

    (SELECT 'Best Weekday', STRFTIME(date, '%A'), ROUND(AVG(ig_engagement_total), 0), COUNT(*)
    FROM posts
    GROUP BY STRFTIME(date, '%A')
    ORDER BY 3 DESC
    LIMIT 1)
""").df()

,dimension,top_performer,avg_engagement,n
0,Best Format,Video,63.0,189
1,Best Content Type,BWI,82.0,7
2,Best Weekday,Sunday,56.0,39


**Video** and **Sunday** are solid results. **BWI** as "best content type" rests on just 7 posts, so I'd read **Educational** (61 over 65 posts) as the more dependable winner.

## 4. Collaborator Analysis

Section 3.5 showed that top posts usually feature a guest. BWI's sheet doesn't have a collaborator column, but the team names them inside the content title (e.g. *"Podcast - Leverage Talk Collab w/ Edward Santoso"*). This section pulls those names out with SQL.

### 4.1 Finding collaboration posts

First, every title that mentions `collab`, `w/` or `w./`.

In [43]:
con.sql("""
    SELECT content
    FROM posts
    WHERE content ILIKE '%collab%' OR content ILIKE '%w/%' OR content ILIKE '%w./%'
    ORDER BY content
""").df()

,content
0,10/12/25 Tridom x BWI C.W/ Wilson & Ronald
1,10/14/25 MM Training JKT Promo 4 Collab W/ TS
2,11/6/2025 Strength Training Promo Collab w/ TS 1
3,11/6/2025 Strength Training Promo Collab w/ TS 2
4,11/6/2025 Strength Training Promo Collab w/ TS 3
...,...
106,Selling skills recap C. W./ Dirga
107,Selling skills recap C. W./ Dirga
108,TS Book Photography BTS w./ Harry
109,Think Like an owner talk w/ Muku


111 posts. Printing every title lets me see all the ways the team wrote names, which shapes the extraction rules:

In [44]:
collab_posts = con.sql("""
    SELECT content
    FROM posts
    WHERE content ILIKE '%collab%' OR content ILIKE '%w/%' OR content ILIKE '%w./%'
    ORDER BY content
""").df()

for c in collab_posts['content']:
    print(c)

10/12/25 Tridom x BWI C.W/ Wilson & Ronald
10/14/25 MM Training JKT Promo 4 Collab W/ TS
11/6/2025 Strength Training Promo Collab w/ TS 1
11/6/2025 Strength Training Promo Collab w/ TS 2
11/6/2025 Strength Training Promo Collab w/ TS 3
11/8/26 5-Day MBA Promo post Collab w/ Muku
11/8/26 5-Day MBA Promo post Collab w/ Muku
11/8/26 5-Day MBA Promo post Collab w/ Ronald
11/8/26 5-Day MBA Promo post Collab w/ Ronald
11/8/26 5-Day MBA Promo post Collab w/ Ronald
11/8/26 5-Day MBA Promo post Collab w/ TS
11/8/26 5-Day MBA Promo post Collab w/ TS
11/8/26 5-Day MBA Promo post Collab w/ Thoni
11/9/25 STBC x BWI w/ Risa
14/10/25 MM JKT Recap Collab W/ Ronald x TS x Muku
14/7/25 5-Day MBA JKT Photos W/ Hary
14/7/25 5-Day MBA JKT Promo 2 Collab W/ TS
14/7/25 5-Day MBA JKT Promo 4 Collab W/Andy
14/7/25 5-Day MBA JKT Promo 5 Collab W/ Ronald
14/7/25 5-Day MBA JKT Promo 6 Collab W/ Fathoni
14/7/25 5-Day MBA JKT Promo 7 Collab W/ TS
18/6/2026 Gen-Z Productivity Promo Vid C. W./ Selly
18/6/2026 Gen-Z P

What the full list shows:
- names follow `w/`, `W/`, `w./` or `C. W./`;
- one post says **`w/o Collab`** ("without collab"), which must be excluded;
- several names can share one post, joined by `&`, `+`, ` x ` or `c./`;
- some numbers are post numbers, not names (`TS 1`, `TS 2`, `TS 3`);
- some "names" aren't people at all (`Past Review`, `Recap`, `storytelling`, `3+ People`);
- spelling varies for the same person (`Hary`/`Harry`, `Thoni`/`Fathoni`, `Ron`/`Ronald`).

### 4.2 Extracting the text after "w/"

The regex `(?i)w\.?\s*/\s*(.+)$` matches `w`, an optional dot, a slash, and captures everything after it. `(?i)` makes it case-insensitive, and `NOT ILIKE '%w/o%'` drops the "without collab" post.

In [45]:
con.sql("""
    SELECT
        content,
        REGEXP_EXTRACT(content, '(?i)w\\.?\\s*/\\s*(.+)$', 1) AS collaborator_raw
    FROM posts
    WHERE (content ILIKE '%w/%' OR content ILIKE '%w./%')
      AND content NOT ILIKE '%w/o%'
    ORDER BY content
""").df()

,content,collaborator_raw
0,10/12/25 Tridom x BWI C.W/ Wilson & Ronald,Wilson & Ronald
1,10/14/25 MM Training JKT Promo 4 Collab W/ TS,TS
2,11/6/2025 Strength Training Promo Collab w/ TS 1,TS 1
3,11/6/2025 Strength Training Promo Collab w/ TS 2,TS 2
4,11/6/2025 Strength Training Promo Collab w/ TS 3,TS 3
...,...,...
104,Selling skills recap C. W./ Dirga,Dirga
105,Selling skills recap C. W./ Dirga,Dirga
106,TS Book Photography BTS w./ Harry,Harry
107,Think Like an owner talk w/ Muku,Muku


Listing the distinct captured strings to see what still needs cleaning:

In [46]:
distinct_collabs = con.sql("""
    SELECT DISTINCT REGEXP_EXTRACT(content, '(?i)w\\.?\\s*/\\s*(.+)$', 1) AS collaborator_raw
    FROM posts
    WHERE (content ILIKE '%w/%' OR content ILIKE '%w./%')
      AND content NOT ILIKE '%w/o%'
    ORDER BY collaborator_raw
""").df()

for c in distinct_collabs['collaborator_raw']:
    print(c)

3+
3+ People
@Suvictor x BWI
Andy
Andy 
Andy & Muku
BWI Internal Team
Dirga
Dr. Lawrence
Edward Santoso 
Fathoni
Harry
Hary
Ivi
Kentgkunkwan
Lisdawaty c./ Ronald
Monica
Muku
Past Review
Recap
Recap past event
Risa
Risa & Ron
Ronald
Ronald x TS x Muku
Selly
TS
TS & Wilson Liem
TS + Ronald
TS 1
TS 2
TS 3
Thoni
Willy
Wilson
Wilson & Ronald
Yuli & TS
Yuliana & Ronald
Yunus
storytelling


### 4.3 Splitting posts with several collaborators

Two nested `REGEXP_REPLACE` calls:
1. remove a trailing ` x BWI` (BWI itself isn't a collaborator);
2. turn every other separator (` x `, `+`, `c./`) into `&`.

Then `STRING_SPLIT` on `&` plus `UNNEST` gives one row per collaborator per post, the same bridge-table idea as the content types.

In [47]:
con.sql("""
    CREATE TABLE collab_raw AS
    SELECT
        raw_row_id,
        REGEXP_EXTRACT(content, '(?i)w\\.?\\s*/\\s*(.+)$', 1) AS collaborator_raw
    FROM posts
    WHERE (content ILIKE '%w/%' OR content ILIKE '%w./%')
      AND content NOT ILIKE '%w/o%'
""")

con.sql("""
    CREATE TABLE collab_split AS
    SELECT
        raw_row_id,
        TRIM(UNNEST(STRING_SPLIT(
            REGEXP_REPLACE(
                REGEXP_REPLACE(collaborator_raw, '(?i)\\s*x\\s*BWI\\s*$', ''),
                '(?i)\\s+x\\s+|\\s*\\+\\s*|\\s*c\\./\\s*', '&', 'g'
            ),
            '&'
        ))) AS name_raw
    FROM collab_raw
""")

con.sql("SELECT DISTINCT name_raw FROM collab_split ORDER BY name_raw").df()

,name_raw
0,
1,3
2,@Suvictor
3,Andy
4,BWI Internal Team
5,Dirga
6,Dr. Lawrence
7,Edward Santoso
8,Fathoni
9,Harry


### 4.4 Standardising names

The final clean-up step:
- `LTRIM(name, '@')` removes the `@` from Instagram handles;
- a regex drops trailing post numbers (`TS 1` → `TS`);
- a `CASE` merges spelling variants into one name;
- a `NOT IN` list removes leftovers that aren't people (blanks, `3`, `People`, `Recap`, etc.).

In [48]:
con.sql("""
    CREATE TABLE collab_final AS
    SELECT
        raw_row_id,
        CASE
            WHEN name = 'Hary' THEN 'Harry'
            WHEN name = 'Thoni' THEN 'Fathoni'
            WHEN name = 'Ron' THEN 'Ronald'
            ELSE name
        END AS collaborator
    FROM (
        SELECT
            raw_row_id,
            TRIM(REGEXP_REPLACE(LTRIM(name_raw, '@'), '\\s*\\d+$', '')) AS name
        FROM collab_split
    )
    WHERE name NOT IN ('', '3', 'People', 'BWI Internal Team', 'Past Review', 'Recap', 'Recap past event', 'storytelling')
""")

con.sql("SELECT DISTINCT collaborator FROM collab_final ORDER BY collaborator").df()

,collaborator
0,Andy
1,Dirga
2,Dr. Lawrence
3,Edward Santoso
4,Fathoni
5,Harry
6,Ivi
7,Kentgkunkwan
8,Lisdawaty
9,Monica


22 clean collaborator names.

### 4.5 Collaborator performance

Average reach and engagement for every post each collaborator appears in.

In [49]:
con.sql("""
    SELECT
        cf.collaborator,
        COUNT(*) AS num_posts,
        ROUND(AVG(p.ig_account_reach), 0) AS avg_reach,
        ROUND(AVG(p.ig_engagement_total), 0) AS avg_engagement
    FROM collab_final cf
    JOIN posts p ON cf.raw_row_id = p.raw_row_id
    GROUP BY cf.collaborator
    ORDER BY avg_engagement DESC
""").df()

,collaborator,num_posts,avg_reach,avg_engagement
0,Ivi,3,3380.0,416.0
1,Dr. Lawrence,2,6659.0,400.0
2,Yuli,1,1813.0,288.0
3,Edward Santoso,3,2348.0,139.0
4,Risa,7,2295.0,114.0
5,TS,25,3721.0,99.0
6,Suvictor,1,4340.0,88.0
7,Harry,3,1295.0,78.0
8,Monica,3,428.0,75.0
9,Dirga,2,0.0,55.0


Ivi (416), Dr. Lawrence (400) and Yuli (288) top this list, but each has only 1-3 posts, and Ivi's figure is lifted by the 1,112-engagement Negotiation Recap. Two other cautions:
- **Dirga shows 0 reach** on both posts, which is almost certainly a missing value logged as 0 rather than a real result;
- Andy has the highest reach (7,813) but low engagement (32), which may reflect boosting.

So I filter to collaborators with **at least 5 posts**:

In [50]:
con.sql("""
    SELECT
        cf.collaborator,
        COUNT(*) AS num_posts,
        ROUND(AVG(p.ig_account_reach), 0) AS avg_reach,
        ROUND(AVG(p.ig_engagement_total), 0) AS avg_engagement
    FROM collab_final cf
    JOIN posts p ON cf.raw_row_id = p.raw_row_id
    GROUP BY cf.collaborator
    HAVING COUNT(*) >= 5
    ORDER BY avg_engagement DESC
""").df()

,collaborator,num_posts,avg_reach,avg_engagement
0,Risa,7,2295.0,114.0
1,TS,25,3721.0,99.0
2,Wilson,6,680.0,43.0
3,Ronald,26,1247.0,43.0
4,Yuliana,6,1059.0,42.0
5,Muku,8,627.0,33.0


With enough posts to compare:
- **Risa** (*114 avg over 7 posts*) and **TS** (*99 over 25, reach 3,721*) are clearly BWI's strongest collaborators. TS in particular combines high engagement with the most posts.
- **Ronald** appears just as often as TS (26 posts) but averages less than half the engagement (43).
- Wilson, Yuliana and Muku sit in the 33 - 43 range.

## 5. Key Findings & Limitations

**Findings**
1. **Video is the strongest format by a wide margin:** about 3× the engagement and 3.5× the reach of carousels, over equal sample sizes (189 each).
2. **Educational content is the most reliable category** (61 avg over 65 posts). Promotional posts get the most reach but not the engagement to match; Greetings and Conversational posts earn almost nothing.
3. **Sunday and Wednesday** get the most engagement, while Tuesday and Friday get the most reach.
4. **Engagement is event-driven:** monthly averages spike around key seminars (May 2025, March 2026, September 2026) and fall back after.
5. **Collaborators matter.** Most top posts feature a guest; among regular collaborators, **TS and Risa** lift engagement the most.

**Limitations**
- **Totals, not rates.** These averages are raw engagement counts, so posts that reach more people naturally score higher. My 2022 - 2025 notebook used *engagement rate* (engagement ÷ reach); repeating that here would show whether video's lead holds once reach is accounted for.
- **Averages are sensitive to outliers.** One post (Negotiation Recap, 1,112) moves Documentation and Ivi's figures noticeably; medians would be a useful check.
- **Boosted posts are included.** The 10 paid boosts inflate reach; comparing results with and without them is a natural next step.
- **Small groups.** BWI-tagged (7), Conversational (3) and several collaborators (1-3 posts) are too small to draw conclusions from.
- **Facebook data is empty** for the whole period, so this covers Instagram only.
- **Dirga's 0 reach** looks like missing data recorded as 0.

## 6. Export

The three clean tables are saved as CSVs so they can feed a dashboard or a later notebook.

In [51]:
con.sql("SELECT * FROM posts").df().to_csv("posts_clean.csv", index=False)
con.sql("SELECT * FROM post_types").df().to_csv("post_types.csv", index=False)
con.sql("SELECT * FROM collab_final").df().to_csv("collab_final.csv", index=False)

print("done")

done


Checking the files are in the folder:

In [52]:
import os
print([f for f in os.listdir() if f.endswith('.csv')])

['cleaned_content_data.csv', 'collab_final.csv', 'Content Analytics Updated - Analytics.csv', 'monthly_ads.csv', 'posts_clean.csv', 'post_types.csv']
